In [ ]:
import pymorphy3
from collections import Counter
import re

morph = pymorphy3.MorphAnalyzer()

def load_text(file_path):
    """Загружает текст из файла."""
    with open(file_path, 'r', encoding='ANSI') as file:
        return file.read()

def extract_sentences_with_ids(text):
    """Извлекает предложения из текста с сохранением их нумерации."""
    sentences = {}
    current_sentence = []
    current_id = None

    for line in text.splitlines():
        match = re.match(r'^\s*(\d+)\s', line)
        if match:
            if current_sentence and current_id is not None:
                sentences[current_id] = ' '.join(current_sentence)
            current_id = int(match.group(1))
            current_sentence = []

        tokens = re.findall(r'<w>.*?<ana.*?/>(.*?)</w>', line)
        if tokens:
            current_sentence.extend(tokens)

    if current_sentence and current_id is not None:
        sentences[current_id] = ' '.join(current_sentence)

    return sentences

def search_by_token(sentences, token):
    """Ищет контексты по заданному токену."""
    result = {sid: sentence for sid, sentence in sentences.items() if token in sentence}
    return result, len(result)

def search_by_lemma(sentences, lemma):
    """Ищет контексты по заданной лемме."""
    result = {}
    for sid, sentence in sentences.items():
        words = sentence.split()
        for word in words:
            parsed = morph.parse(word)[0]
            if parsed.normal_form == lemma:
                result[sid] = sentence
                break
    return result, len(result)

def search_by_semantic_tag(text, tag):
    """Ищет контексты по заданному семантическому тегу."""
    sentences = {}
    current_sentence = []
    current_id = None
    tag_found = False

    for line in text.splitlines():
        match = re.match(r'^\s*(\d+)\s', line)
        if match:
            if current_sentence and current_id is not None and tag_found:
                sentences[current_id] = ' '.join(current_sentence)
            current_id = int(match.group(1))
            current_sentence = []
            tag_found = False

        if re.search(rf'<ana [^>]*{re.escape(tag)}', line):
            tag_found = True

        tokens = re.findall(r'<w>.*?<ana.*?/>(.*?)</w>', line)
        if tokens:
            current_sentence.extend(tokens)

    if current_sentence and current_id is not None and tag_found:
        sentences[current_id] = ' '.join(current_sentence)

    return sentences, len(sentences)

def search_by_tag_combination(text, tags):
    """Ищет контексты по заданной комбинации тегов."""
    sentences = {}
    current_sentence = []
    current_id = None
    tags_found = set()

    for line in text.splitlines():
        match = re.match(r'^\s*(\d+)\s', line)
        if match:
            if current_sentence and current_id is not None and tags_found == set(tags):
                sentences[current_id] = ' '.join(current_sentence)
            current_id = int(match.group(1))
            current_sentence = []
            tags_found = set()

        for tag in tags:
            if re.search(rf'<ana [^>]*{re.escape(tag)}', line):
                tags_found.add(tag)

        tokens = re.findall(r'<w>.*?<ana.*?/>(.*?)</w>', line)
        if tokens:
            current_sentence.extend(tokens)

    if current_sentence and current_id is not None and tags_found == set(tags):
        sentences[current_id] = ' '.join(current_sentence)

    return sentences, len(sentences)

def count_frequencies(sentences, text):
    """Подсчитывает частотность токенов, лемм, тегов и комбинаций тегов."""
    tokens = []
    lemmas = []
    tags = []
    combinations = []

    for sentence in sentences.values():
        words = re.findall(r'\b\w+\b', sentence)
        tokens.extend(words)
        lemmas.extend([morph.parse(word)[0].normal_form for word in words])

    for line in text.splitlines():
        tags.extend(re.findall(r'gr=\'(.*?)\'|sem=\'(.*?)\'', line))
        combinations.extend(re.findall(r'gr=\'(.*?)\' sem=\'(.*?)\'', line))

    token_count = len(tokens)
    lemma_count = len(lemmas)
    tag_count = len(tags)
    combination_count = len(combinations)

    return token_count, lemma_count, tag_count, combination_count

def save_summary(token_count, lemma_count, tag_count, combination_count, file_name):
    """Сохраняет итоговую информацию в файл."""
    with open(file_name, 'w', encoding='ANSI') as file:
        file.write(f"Токенов: {token_count}\n")
        file.write(f"Лемм: {lemma_count}\n")
        file.write(f"Тегов: {tag_count}\n")
        file.write(f"Комбинаций тегов: {combination_count}\n")

def save_contexts(contexts, count, file_name):
    """Сохраняет контексты и их количество в файл."""
    with open(file_name, 'w', encoding='ANSI') as file:
        file.write(f"Количество: {count}\n\n")
        for sid, context in contexts.items():
            file.write(f"{sid}: {context}\n")

# Основной код
file_path = 'C:/Users/user/Documents/GitHub/MSP/RNC_Subcorpus/instrumenty/metla.txt'
text = load_text(file_path)
sentences = extract_sentences_with_ids(text)

# Задание а: Поиск по токену
token = 'метлой'
token_contexts, token_count = search_by_token(sentences, token)
save_contexts(token_contexts, token_count, 'token_results.txt')

# Задание б: Поиск по лемме
lemma = 'метла'
lemma_contexts, lemma_count = search_by_lemma(sentences, lemma)
save_contexts(lemma_contexts, lemma_count, 'lemma_results.txt')

# Задание в: Поиск по семантическому тегу
tag = 'r:spec'
tag_contexts, tag_count = search_by_semantic_tag(text, tag)
save_contexts(tag_contexts, tag_count, 'tag_results.txt')

# Задание г: Поиск по комбинации тегов
tags = ['r:concr', 't:tool:instr']
combination_contexts, combination_count = search_by_tag_combination(text, tags)
save_contexts(combination_contexts, combination_count, 'combination_results.txt')

# Задание д: Подсчёт частотности токенов, лемм, тегов и комбинаций тегов
token_count, lemma_count, tag_count, combination_count = count_frequencies(sentences, text)
save_summary(token_count, lemma_count, tag_count, combination_count, 'summary.txt')

print("Операции завершены. Результаты сохранены в файлы.")


Операции завершены. Результаты сохранены в файлы.
